# 03 — Heston–Merton playground (Monte Carlo)

Synthetic paths only — no market data.

$$dv_t = \kappa(\theta - v_t)\,dt + \xi\sqrt{v_t}\,dW^v_t$$
$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa_J)\,dt + \sqrt{v_t}\,dW^S_t + (e^J-1)\,dN_t$$

with \(\mathrm{Corr}(dW^S, dW^v)=\rho\). Euler–Maruyama with \(v \leftarrow \max(v,0)\).

Try high \(\xi\) (vol-of-vol) or low \(\kappa\) (slow mean reversion) to see stochastic-vol clustering; raise \(\lambda\) for jumps.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_heston_merton(
    mu, kappa, theta, xi, rho, v0,
    lam, mu_j, sigma_j,
    S0, T, n_steps, n_paths, seed=42,
):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    kappa_j = np.exp(mu_j + 0.5 * sigma_j**2) - 1.0

    S = np.full(n_paths, S0, dtype=float)
    v = np.full(n_paths, v0, dtype=float)
    paths = np.empty((n_paths, n_steps + 1))
    vol_paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = S
    vol_paths[:, 0] = np.sqrt(np.maximum(v, 0.0))
    log_rets = np.empty((n_paths, n_steps))

    for i in range(n_steps):
        z1 = rng.standard_normal(n_paths)
        z2 = rng.standard_normal(n_paths)
        dWs = z1
        dWv = rho * z1 + np.sqrt(max(1.0 - rho**2, 0.0)) * z2

        v_pos = np.maximum(v, 0.0)
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * dWv
        v = np.maximum(v, 0.0)

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump = np.zeros(n_paths)
        mask = n_jumps > 0
        jump[mask] = (
            n_jumps[mask] * mu_j
            + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
        )

        incr = (mu - 0.5 * v_pos - lam * kappa_j) * dt + np.sqrt(v_pos * dt) * dWs + jump
        S = S * np.exp(incr)
        paths[:, i + 1] = S
        vol_paths[:, i + 1] = np.sqrt(v)
        log_rets[:, i] = incr

    t = np.linspace(0, T, n_steps + 1)
    return t, paths, vol_paths, log_rets

def plot_heston_merton(
    mu=0.05, kappa=2.0, theta=0.04, xi=0.5, rho=-0.6, v0=0.04,
    lam=0.3, mu_j=-0.05, sigma_j=0.10,
    S0=100.0, T=1.0, n_steps=252, n_paths=40,
):
    t, paths, vol_paths, log_rets = simulate_heston_merton(
        mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.8)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean")
    axes[0].set_title("Price paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].plot(t, vol_paths.T, alpha=0.35, lw=0.8)
    axes[1].plot(t, vol_paths.mean(axis=0), color="black", lw=2)
    axes[1].set_title("Instantaneous vol √v")
    axes[1].set_xlabel("years")

    axes[2].hist(log_rets.ravel(), bins=80, density=True, alpha=0.75, color="seagreen")
    axes[2].set_title("Step log returns")
    axes[2].set_xlabel("log return")

    fig.suptitle(
        f"κ={kappa:.1f}, θ={theta:.3f}, ξ={xi:.2f}, ρ={rho:.2f}, λ={lam:.2f}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_heston_merton,
    mu=FloatSlider(value=0.05, min=-0.10, max=0.30, step=0.01, description="μ"),
    kappa=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description="κ"),
    theta=FloatSlider(value=0.04, min=0.005, max=0.20, step=0.005, description="θ var"),
    xi=FloatSlider(value=0.50, min=0.05, max=2.0, step=0.05, description="ξ volvol"),
    rho=FloatSlider(value=-0.60, min=-0.95, max=0.95, step=0.05, description="ρ"),
    v0=FloatSlider(value=0.04, min=0.005, max=0.20, step=0.005, description="v0"),
    lam=FloatSlider(value=0.30, min=0.0, max=3.0, step=0.1, description="λ"),
    mu_j=FloatSlider(value=-0.05, min=-0.30, max=0.15, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.10, min=0.01, max=0.40, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="T"),
    n_steps=IntSlider(value=252, min=50, max=750, step=10, description="steps"),
    n_paths=IntSlider(value=40, min=5, max=120, step=5, description="paths"),
);

interactive(children=(FloatSlider(value=0.05, description='μ', max=0.3, min=-0.1, step=0.01), FloatSlider(valu…